# Python lab: Asset Returns — Empirical Stylized Facts

ใช้ Python standard library กด Run All ตามลำดับ ข้อมูลทั้งหมดจำลองด้วย seed ที่ระบุ ภาพประกอบฝังอยู่ในไฟล์แล้ว

# Asset Returns — Empirical Stylized Facts

ราคาพรุ่งนี้อาจเดาทิศทางได้ยาก แต่ขนาดการแกว่งพรุ่งนี้มีข้อมูลจากวันนี้อยู่แค่ไหน?

> “การเปลี่ยนแปลงขนาดใหญ่มักตามด้วยการเปลี่ยนแปลงขนาดใหญ่ ไม่ว่าจะขึ้นหรือลง ส่วนการเปลี่ยนแปลงขนาดเล็กมักตามด้วยการเปลี่ยนแปลงขนาดเล็ก”
>
> <span lang="en">“…large changes tend to be followed by large changes—of either sign—and small changes tend to be followed by small changes…”</span>
>
> — **Benoit Mandelbrot** · [*The Variation of Certain Speculative Prices* (1963), น. 418](https://oftp.cyrax.hu/doc/mandelbrot.pdf#page=26) · ข้อความบางส่วน แปลไทยเพื่อประกอบบทเรียน

ในบท [VaR และ Expected Shortfall](../value-at-risk-expected-shortfall.html) เราคำนวณขาดทุนจากการแจกแจงที่กำหนดให้แล้ว ถ้าเลือก Normal และใช้ volatility คงที่ สูตรจะคำนวณได้สะดวก แต่วันที่ตลาดผันผวนมากมักเกิดติดกัน ช่วงหนึ่งของข้อมูลจึงอาจมีขนาดความเสี่ยงต่างจากอีกช่วงอย่างชัดเจน

**Stylized facts** คือรูปแบบเชิงสถิติที่พบซ้ำในข้อมูลหลายตลาดและหลายช่วงเวลา เช่น หางของการแจกแจงที่หนากว่า Normal หรือความสัมพันธ์ของขนาดผลตอบแทนระหว่างวัน เป็นข้อสังเกตที่ใช้ตรวจและเลือกแบบจำลอง ไม่ใช่กฎที่สินทรัพย์ทุกตัวต้องทำตามในทุกความถี่

บทนี้ใช้เส้นเรื่องจากงานของ Stephen Taylor เรื่อง *Asset Price Dynamics, Volatility, and Prediction* โดยเริ่มจากผลตอบแทนรายวันแล้วลงไปดูราคาภายในวัน กราฟและตัวทดลองทั้งหมดคำนวณขึ้นใหม่จากข้อมูลสมมติ เพื่อแยกผลของแต่ละสมมติฐาน ไม่มีกราฟใดเป็นราคาตลาดจริงหรือการทดสอบกลยุทธ์ลงทุน

In [1]:
"""Standard-library numerical examples; all observations are simulated."""
import math
import statistics


def uniform(seed):
    state = seed & 0xffffffff
    while True:
        state = (1664525*state + 1013904223) & 0xffffffff
        yield (state+.5)/4294967296


def normal_generator(seed):
    u = uniform(seed)
    while True:
        yield math.sqrt(-2*math.log(next(u)))*math.cos(2*math.pi*next(u))


def return_pair(previous, current, dividend=0):
    assert previous > 0 and current+dividend > 0
    simple = (current+dividend)/previous-1
    return simple, math.log1p(simple)


def moments(values):
    average = statistics.mean(values)
    centered = [x-average for x in values]
    m2 = statistics.mean(x*x for x in centered)
    m4 = statistics.mean(x**4 for x in centered)
    return {'mean': average, 'sd': statistics.stdev(values), 'variance': m2,
            'kurtosis': m4/m2**2 if m2 > 0 else None}


def acf(values, max_lag=20):
    assert 0 <= max_lag < len(values)
    average = statistics.mean(values)
    x = [v-average for v in values]
    denominator = sum(v*v for v in x)
    if denominator == 0:
        return [None]*(max_lag+1)
    return [sum(x[i]*x[i-lag] for i in range(lag,len(x)))/denominator for lag in range(max_lag+1)]


def portmanteau(values, lags=20):
    rho = acf(values,lags)
    if rho[0] is None:
        return None, None
    n = len(values)
    return n*sum(r*r for r in rho[1:]), n*(n+2)*sum(rho[k]**2/(n-k) for k in range(1,lags+1))


def shuffle(values, seed=731):
    out = list(values)
    random = uniform(seed)
    for i in range(len(out)-1,0,-1):
        j = math.floor(next(random)*(i+1))
        out[i],out[j] = out[j],out[i]
    return out


def clustered_returns(seed=2524):
    random = normal_generator(seed)
    return [next(random)*(.005 if (i//50)%2 == 0 else .025) for i in range(600)]


def variance_mixture(p=.2, ratio=5):
    assert 0 <= p <= 1 and ratio >= 1
    variance = 1-p+p*ratio**2
    sd = math.sqrt(variance)
    low, high = 1/sd, ratio/sd
    normal = statistics.NormalDist()
    return {'variance': variance, 'sd': sd, 'low': low, 'high': high,
            'kurtosis': 3*((1-p)+p*ratio**4)/variance**2,
            'density': lambda x: (1-p)*normal.pdf(x/low)/low+p*normal.pdf(x/high)/high,
            'tail': lambda threshold: (1-p)*math.erfc(abs(threshold)/low/math.sqrt(2))+p*math.erfc(abs(threshold)/high/math.sqrt(2))}


def realized_variance(log_prices, stride=1):
    assert isinstance(stride,int) and stride > 0 and (len(log_prices)-1)%stride == 0
    returns = [log_prices[i]-log_prices[i-stride] for i in range(stride,len(log_prices),stride)]
    variance = sum(r*r for r in returns)
    return {'variance':variance, 'volatility':math.sqrt(variance), 'returns':returns, 'count':len(returns)}


def intraday_sample(noise_bps=3, seed=81):
    assert 0 <= noise_bps <= 10
    random, signs = normal_generator(seed), uniform(seed+1000)
    latent = [0]
    for _ in range(390):
        latent.append(latent[-1]+.01/math.sqrt(390)*next(random))
    eta = noise_bps/10000
    observed = [p+eta*(-1 if next(signs)<.5 else 1) for p in latent]
    return {'latent':latent, 'observed':observed, 'eta':eta, 'integrated_variance':.01**2}


def intraday_profile(news=True):
    raw = [1+3*math.exp(-i/6)+2*math.exp(-(77-i)/7)+(5*math.exp(-.5*((i-30)/1.2)**2) if news else 0) for i in range(78)]
    total = sum(raw)
    return [x/total for x in raw]


def close(a,b,tol=1e-10):
    assert math.isclose(a,b,rel_tol=tol,abs_tol=tol),(a,b)
print("Loaded self-contained functions. Simulated data only; no remote downloads.")

Loaded self-contained functions. Simulated data only; no remote downloads.


## จากราคาไปเป็นผลตอบแทน

ให้ \(P_{t-1}\) เป็นราคาก่อนเริ่มช่วง \(P_t\) เป็นราคาปลายช่วง และ \(D_t\) เป็นเงินปันผลที่ได้รับเมื่อสิ้นช่วงต่อหนึ่งหน่วยสินทรัพย์ โดยยังไม่หักต้นทุนซื้อขาย

$$
R_t=\frac{P_t+D_t-P_{t-1}}{P_{t-1}},\qquad
r_t=\log(1+R_t)=\log\left(\frac{P_t+D_t}{P_{t-1}}\right).
$$

R คือ simple return ส่วน r คือ log return ตัวอย่างซื้อที่ 100 ขายที่ 99 และรับปันผล 2 จะได้ R=1% และ r≈0.9950% ถ้าใช้ราคาดิบแล้วละปันผล เราจะคำนวณ R เป็น −1% ทั้งที่ความมั่งคั่งรวมเพิ่มขึ้น

เมื่อ R ใกล้ศูนย์ ค่า r ใกล้ R แต่ความต่างเพิ่มตามขนาดการเปลี่ยนแปลง การขึ้น 20% แล้วลง 20% ให้ simple return สะสม −4% ส่วน log returns บวกกันได้ \(\log(1.2)+\log(0.8)=\log(0.96)\) ไม่ใช่ศูนย์

$$
R_{1:T}=\prod_{t=1}^{T}(1+R_t)-1,\qquad
r_{1:T}=\sum_{t=1}^{T}r_t=\log(1+R_{1:T}).
$$

ถ้ามีปันผลระหว่างหลายช่วง ผลตอบแทนทบต้นนี้ใช้ wealth index ที่นำปันผลกลับไปลงทุนตามสมมติฐานเดียวกัน สำหรับข้อมูลที่ปรับปันผลและแตกหุ้นไว้แล้ว ต้องตรวจวิธี adjustment ก่อนบวกปันผลซ้ำ

บทนี้ใช้ **log return** สำหรับอนุกรมเวลาและ realized variance เป็นหลัก ส่วนการรวมสินทรัพย์ด้วยน้ำหนักพอร์ต ณ ต้นช่วงยังใช้ simple returns: \(R_p=\sum_iw_iR_i\) การเฉลี่ย log returns ของสินทรัพย์ด้วยน้ำหนักเดียวกันไม่ได้ให้ log return ของพอร์ตตรง ๆ

In [2]:
simple, log_return = return_pair(100,99,2)
close(simple,.01)
close(log_return,math.log(1.01))
print(f"Dividend-inclusive simple return {simple:.6%}; log return {log_return:.6%}")
compound = 1.2*.8-1
sum_log = math.log(1.2)+math.log(.8)
close(math.expm1(sum_log),compound)
print(f"+20%, then -20%: simple {compound:.4%}; log {sum_log:.4%}")

Dividend-inclusive simple return 1.000000%; log return 0.995033%
+20%, then -20%: simple -4.0000%; log -4.0822%


## รูปแบบสามอย่างในผลตอบแทนรายวัน

Taylor จัดข้อสังเกตหลักของผลตอบแทนรายวันไว้สามข้อ การอ่านแต่ละข้อควรระบุทั้งสินทรัพย์ ช่วงข้อมูล และความถี่ที่ใช้

| สิ่งที่พบซ้ำ | ดูจากอะไร | สมมติฐานที่ต้องตรวจต่อ |
|---|---|---|
| การแจกแจงมีหางหนากว่า Normal | Histogram, Q–Q plot, tail probabilities และ kurtosis | Normal ที่ใช้ประมาณขาดทุนครอบคลุมหางหรือไม่ |
| ผลตอบแทนคนละวันมักมี linear correlation ต่ำ | Scatter plot และ ACF ของ r | Correlation ต่ำไม่ได้รับรอง independence |
| ขนาดผลตอบแทนมีความสัมพันธ์บวกข้ามวัน | ACF ของ \(|r|\) และ \(r^2\) | Volatility คงที่อาจไม่เหมาะกับข้อมูล |

ผลตอบแทนหุ้นยังอาจเบ้ด้านขาดทุน ขนาดความเบ้และความหนาของหางเปลี่ยนตามตลาดและช่วงตัวอย่าง ส่วนคำว่า correlation ต่ำไม่ได้หมายถึงเท่ากับศูนย์ทุก lag โดยเฉพาะสินทรัพย์ที่ซื้อขายบางหรือข้อมูลระยะสั้นมาก

ค่าเฉลี่ยรายวันมักมีขนาดเล็กเมื่อเทียบกับ SD การนำ mean จากหน้าต่างสั้นไป annualize จึงทำให้ค่าประมาณที่คลาดเคลื่อนดูใหญ่ขึ้นได้ ส่วน SD เองก็เปลี่ยนตามช่วงที่เลือก ถ้าหน้าต่างมีช่วงวิกฤตรวมอยู่ ค่าที่ประมาณย่อมต่างจากหน้าต่างสงบ

เอกสารเชิงประจักษ์เคยรายงาน calendar effects ทั้งวันในสัปดาห์ ต้นเดือน มกราคม และก่อนวันหยุด แต่ผลที่พบในข้อมูลหนึ่งชุดอาจอ่อนลงเมื่อเปลี่ยนช่วงเวลา การทดสอบหลายปฏิทินย้อนหลังยังเสี่ยงเลือกเฉพาะผลที่บังเอิญเด่น จึงต้องตรวจข้อมูลนอกช่วงประมาณค่าและต้นทุนซื้อขายก่อนนำไปใช้

## วันที่แกว่งแรงมักอยู่ใกล้กัน

[Volatility clustering](../glossary.html#volatility-clustering) หมายถึงช่วงที่ผลตอบแทนมีขนาดใหญ่เกิดติดกัน สลับกับช่วงที่ขนาดเล็ก คำว่า “ขนาดใหญ่” ครอบคลุมทั้งบวกและลบ หลังวันที่ลงแรง วันถัดไปอาจลงต่อหรือดีดกลับแรงก็ได้

แบบจำลองง่าย ๆ ที่แยกสองส่วนนี้คือ

$$
r_t=\mu_t+\sigma_tz_t,\qquad
\mathbb E[z_t\mid\mathcal F_{t-1}]=0,\qquad
\mathbb E[z_t^2\mid\mathcal F_{t-1}]=1.
$$

\(\mathcal F_{t-1}\) คือข้อมูลที่รู้ก่อนเริ่มช่วง t ส่วน \(\mu_t\) และ \(\sigma_t\) เป็นค่าที่กำหนดจากข้อมูลนั้นได้ เมื่อ \(\sigma_t\) สูง ขนาด \(|r_t-\mu_t|\) มีแนวโน้มสูงขึ้น แต่เครื่องหมายยังขึ้นกับช็อก \(z_t\) การคาดการณ์ volatility จึงไม่ได้ให้คำตอบทิศทางผลตอบแทนโดยอัตโนมัติ



ตัวอย่างสมมติใช้ Normal shocks อิสระ และสลับ SD 0.5% กับ 2.5% ทุก 50 วัน ไม่ใช่ผลประมาณ GARCH หรือข้อมูลตลาด การกำหนดช่วงเช่นนี้ทำให้เห็นผลของสเกลที่เปลี่ยนตามเวลาโดยตรง

ถ้าเก็บผลตอบแทนทุกค่าไว้เหมือนเดิมแล้วสับลำดับวัน ค่าเฉลี่ย SD และ histogram จะไม่เปลี่ยน แต่ช่วงผันผวนที่เรียงติดกันจะถูกรบกวน การดู histogram อย่างเดียวจึงแยกข้อมูลสองลำดับนี้ไม่ได้

In [3]:
returns = clustered_returns()
permuted = shuffle(returns)
assert sorted(returns)==sorted(permuted)
before, after = moments(returns),moments(permuted)
for key in before:
    close(before[key],after[key])
for label, values in [('Original',returns),('Shuffled',permuted)]:
    print(label,moments(values))
    print(f"lag 1: r={acf(values)[1]:.6f}, |r|={acf([abs(r) for r in values])[1]:.6f}")
print("The same values in a different order: moments/histogram unchanged.")

Original {'mean': -0.0013171622041231533, 'sd': 0.01700961364251322, 'variance': 0.0002888447446737925, 'kurtosis': 5.025022817831761}
lag 1: r=0.051878, |r|=0.329642
Shuffled {'mean': -0.0013171622041231533, 'sd': 0.01700961364251322, 'variance': 0.0002888447446737925, 'kurtosis': 5.025022817831761}
lag 1: r=-0.059582, |r|=0.073624
The same values in a different order: moments/histogram unchanged.


## วัดความสัมพันธ์ข้ามเวลาด้วย ACF

[Autocorrelation function หรือ ACF](../glossary.html#autocorrelation) วัดความสัมพันธ์ระหว่างค่าที่ห่างกัน k ช่วงเวลา นิยาม sample ACF ที่ใช้ในบทนี้คือ

$$
\widehat\rho_k=
\frac{\sum_{t=k+1}^{n}(r_t-\bar r)(r_{t-k}-\bar r)}
{\sum_{t=1}^{n}(r_t-\bar r)^2},\qquad k=1,\ldots,m.
$$

เราใช้ค่าเฉลี่ยของชุดเต็มและตัวหารชุดเต็มทุก lag นิยามนี้อาจต่างเล็กน้อยจากการใช้ Pearson correlation กับสองช่วงที่ตัดแล้ว เช่นฟังก์ชัน CORREL ที่หา mean ของแต่ละช่วงใหม่ สำหรับข้อมูลคงที่ทุกค่า ตัวหารเป็นศูนย์และ ACF ไม่มีนิยาม

ถ้าค่าต่าง ๆ เป็น iid และมีเงื่อนไขโมเมนต์ที่เหมาะสม กรอบอ้างอิงราย lag สำหรับตัวอย่างขนาดใหญ่ประมาณได้ด้วย \(\pm1.96/\sqrt n\) เช่น n=600 ให้ประมาณ ±0.080 ดูวิธีสร้างกรอบใน [NIST, Autocorrelation Plot](https://www.itl.nist.gov/div898/handbook/eda/section3/eda331.htm)

กรอบนี้เป็นการพิจารณาทีละ lag การดู 20 หรือ 30 lag พร้อมกันเพิ่มโอกาสเห็นจุดหลุดกรอบโดยบังเอิญ และภายใต้ conditional heteroskedasticity การใช้กรอบ iid กับผลตอบแทนดิบอาจให้ inference ที่ไม่เหมาะสม การไม่หลุดกรอบก็ไม่ได้พิสูจน์ว่าเป็นอิสระ

Box–Pierce และ Ljung–Box รวม ACF หลาย lag เป็นสถิติเดียว

$$
Q_{\rm BP}=n\sum_{k=1}^{m}\widehat\rho_k^2,\qquad
Q_{\rm LB}=n(n+2)\sum_{k=1}^{m}\frac{\widehat\rho_k^2}{n-k}.
$$

ภายใต้ null และเงื่อนไขที่เหมาะสมของ white-noise/iid benchmark ค่าสถิติมีการแจกแจงอ้างอิงโดยประมาณแบบ \(\chi_m^2\) หากทดสอบ residuals จากโมเดลที่ประมาณพารามิเตอร์แล้ว องศาอิสระและเงื่อนไขต้องปรับตามโมเดล การปฏิเสธหรือไม่ปฏิเสธด้วยสถิตินี้จึงต้องอ้างอิงโจทย์ที่ตั้งไว้ ไม่ใช่ข้อสรุปกว้าง ๆ ว่า “ทำนายราคาได้”

ลองคำนวณ ACF และ Q อีกครั้งหลังแทน r ด้วย \(|r|\) หรือ \(r^2\) ถ้า r เป็น iid ฟังก์ชันเหล่านี้ก็ควรเป็น iid เช่นกัน ความสัมพันธ์ในขนาดผลตอบแทนจึงเป็นหลักฐานที่ใช้โต้แย้งสมมติฐาน iid ของ r ได้ แม้ ACF ของ r เองจะต่ำ

In [4]:
for label, values in [('r',returns),('|r|',[abs(r) for r in returns]),('r^2',[r*r for r in returns])]:
    bp, lb = portmanteau(values,20)
    print(f"{label}: Box-Pierce Q(20)={bp:.4f}; Ljung-Box Q(20)={lb:.4f}")
print(f"Pointwise iid reference +/-1.96/sqrt(n) = {1.96/math.sqrt(len(returns)):.6f}")
print("No p-values claimed for the heteroskedastic synthetic return series.")
assert acf([3,3,3],1)==[None,None]

r: Box-Pierce Q(20)=43.3992; Ljung-Box Q(20)=44.2310
|r|: Box-Pierce Q(20)=658.4624; Ljung-Box Q(20)=667.8226
r^2: Box-Pierce Q(20)=216.2522; Ljung-Box Q(20)=219.4868
Pointwise iid reference +/-1.96/sqrt(n) = 0.080017
No p-values claimed for the heteroskedastic synthetic return series.


## หางหนาและ kurtosis

สำหรับ Standard Normal โอกาสอยู่ห่างค่าเฉลี่ยเกิน 3 SD ทั้งสองด้านรวมประมาณ 0.2700% และเกิน 4 SD ประมาณ 0.00633% ถ้าสุ่มอิสระวันละหนึ่งค่าตลอด 252 วัน จำนวนเหตุการณ์เกิน 3 SD คาดหมายอยู่ที่ประมาณ 0.68 ครั้งต่อปี ตัวเลขนี้มาจากแบบจำลอง ไม่ใช่อัตราที่ตลาดต้องเกิดจริง

การแจกแจงที่มี [fat tails](../glossary.html#fat-tails) ให้โอกาสเหตุการณ์รุนแรงมากกว่า Normal ที่ใช้เทียบ การตัดข้อมูลวันที่ร่วงแรงออกเพียงเพราะอยู่นอก 3 SD อาจตัดส่วนที่ต้องใช้ประเมินความเสี่ยงออกไปด้วย ต้องแยกข้อมูลผิด เช่นราคา stale หรือ split ที่ยังไม่ปรับ ออกจากราคาที่เคลื่อนไหวแรงจริง

Kurtosis แบบ population คือ

$$
\kappa=\frac{\mathbb E[(r-\mu)^4]}{\operatorname{Var}(r)^2}.
$$

Normal มี κ=3 และ excess kurtosis เท่ากับ κ−3 ในตัวทดลองใช้ moment estimator \(\widehat\kappa=m_4/m_2^2\) โดย \(m_j=n^{-1}\sum_t(r_t-\bar r)^j\) ส่วน SD ที่รายงานใช้ตัวหาร n−1 โปรแกรมที่ปรับ small-sample bias หรือรายงานเฉพาะ excess kurtosis จะให้ตัวเลขคนละนิยาม

Kurtosis ไวต่อข้อมูลปลายหางเพราะยกกำลังสี่ ตัวเลขสูงไม่ได้บอกความเบ้ และไม่ได้ระบุว่า distribution ต้องเป็น Student-t หรือมี variance อนันต์ การทดสอบ normality ด้วย \((\widehat\kappa-3)/\sqrt{24/n}\) อาศัย iid Normal null และการประมาณตัวอย่างใหญ่ จึงไม่ควรใช้ standard error นี้ตรง ๆ กับข้อมูลที่มี volatility clustering

In [5]:
for threshold in [3,4]:
    probability = math.erfc(threshold/math.sqrt(2))
    print(f"Normal two-sided P(|Z|>{threshold})={probability:.8%}; expected per 252 draws={252*probability:.6f}")
print(f"Moment kurtosis: {moments(returns)['kurtosis']:.6f}; excess: {moments(returns)['kurtosis']-3:.6f}")
print("Moment estimator uses m4/m2^2; reported sample SD uses n-1.")

Normal two-sided P(|Z|>3)=0.26997961%; expected per 252 draws=0.680349
Normal two-sided P(|Z|>4)=0.00633425%; expected per 252 draws=0.015962
Moment kurtosis: 5.025023; excess: 2.025023
Moment estimator uses m4/m2^2; reported sample SD uses n-1.


## Normal หลายสเกลรวมกันให้หางหนาได้

สมมติว่าในแต่ละสภาวะผลตอบแทนมีค่าเฉลี่ยศูนย์ และ

$$
r\mid\sigma\sim N(0,\sigma^2).
$$

เมื่อรวมหลายสภาวะเข้าด้วยกัน \(\mathbb E[r^2]=\mathbb E[\sigma^2]\) และ \(\mathbb E[r^4]=3\mathbb E[\sigma^4]\) จึงได้

$$
\kappa=3\frac{\mathbb E[\sigma^4]}{\mathbb E[\sigma^2]^2}
=3\left(1+\frac{\operatorname{Var}(\sigma^2)}{\mathbb E[\sigma^2]^2}\right)\geq3.
$$

ค่า κ จะมากกว่า 3 เมื่อ variance ของแต่ละสภาวะต่างกันจริง และโมเมนต์ที่ใช้มีค่าจำกัด สูตรนี้ใช้ค่าเฉลี่ยร่วมเดียวกัน ไม่ใช่ข้อสรุปสำหรับ mixture ที่ค่าเฉลี่ยของแต่ละกลุ่มต่างกันตามอำเภอใจ

ตัวอย่างให้ 80% ของวันมี SD 0.5% และอีก 20% มี SD 2.5% จะได้ variance รวม \(0.8(0.005)^2+0.2(0.025)^2=0.000145\) หรือ SD ประมาณ 1.2042% ต่อวัน ส่วน kurtosis เท่ากับประมาณ 11.219 เมื่อเทียบกับ Normal ที่มี SD เท่ากัน การต่างกันของรูปหางจึงยังอยู่

Mixture อธิบายการแจกแจงเมื่อรวมข้อมูลหลายสเกล แต่ยังไม่ได้บอกลำดับเวลา เราอาจสุ่มเลือกสภาวะใหม่อย่างอิสระทุกวันแล้วได้ marginal distribution แบบเดียวกัน โดยไม่มี volatility clustering หรือกำหนดให้สภาวะคงอยู่นานจนเกิด clustering ก็ได้ ต้องดูทั้งการแจกแจงและ dependence จึงจะแยกสองกรณีนี้ออก

การเปลี่ยน volatility เป็นกลไกหนึ่งที่สร้างหางหนาได้ ส่วน jumps ความเบ้ และหางของช็อก \(z_t\) อาจเพิ่มความผิดปกติจาก Normal อีก การหารด้วย volatility ที่ประมาณมาแล้วไม่ได้รับประกันว่าข้อมูลที่เหลือจะเป็น Normal

In [6]:
mixture = variance_mixture(.2,5)
variance = .8*.005**2+.2*.025**2
close(variance,.000145)
close(mixture['kurtosis'],11.218787158145064)
close(variance_mixture(.2,1)['kurtosis'],3)
print(f"Daily SD={math.sqrt(variance):.6%}; kurtosis={mixture['kurtosis']:.6f}")
print(f"Two-sided tail beyond 3 pooled SD={mixture['tail'](3):.6%}")
print("Mixture probabilities alone do not specify temporal dependence.")

Daily SD=1.204159%; kurtosis=11.218787
Two-sided tail beyond 3 pooled SD=2.969206%
Mixture probabilities alone do not specify temporal dependence.


## สิ่งที่เปลี่ยนเมื่อขยายช่วงเวลา

เมื่อรวมผลตอบแทนหลายวัน รูปการแจกแจงอาจเข้าใกล้ Normal ภายใต้เงื่อนไข central limit theorem เช่น variance จำกัดและ dependence ที่ไม่รุนแรงเกินไป แต่จำนวนวันเท่าใดจึงใกล้พอขึ้นกับข้อมูล โดยเฉพาะความแม่นที่ต้องการตรงหาง การเห็น monthly kurtosis ใกล้ 3 ไม่ได้พิสูจน์ว่า daily returns เป็น Normal

Volatility clustering ทำให้สมมติฐาน constant-parameter GBM ของบทก่อนครอบคลุมข้อมูลได้ไม่ครบ แบบจำลอง continuous-time ยังมี volatility ที่เปลี่ยนตามเวลาได้ และแบบจำลอง discrete-time อย่าง ARCH/GARCH ออกแบบให้ conditional variance ตอบสนองต่อข้อมูลอดีตได้ รายละเอียดการประมาณโมเดลเหล่านี้เป็นงานต่อจากการตรวจพฤติกรรมข้อมูลในบทนี้

สำหรับ VaR/ES การเปลี่ยนจาก unconditional SD หนึ่งค่าไปใช้ volatility ที่คาดการณ์ ณ วันนั้นจะเปลี่ยนความเสี่ยงที่รายงาน การเลือก distribution ของ standardized shocks ก็ยังมีผลต่อหาง ส่วนการปรับสูตรราคา Option ต้องพิจารณาความน่าจะเป็นสำหรับการตั้งราคาและ volatility risk premium เพิ่มเติม ไม่สามารถนำผลประมาณภายใต้ physical measure ไปแทน risk-neutral dynamics ทั้งชุดทันที

## เมื่อหนึ่งวันมีราคาหลายพันค่า

ข้อมูลซื้อขายระหว่างวันมีทั้งราคา bid, ask และ transaction price ผู้ซื้ออาจซื้อที่ ask ผู้ขายอาจขายที่ bid และบางธุรกรรมเกิดระหว่างสองราคา ระยะห่างระหว่างรายการซื้อขายไม่คงที่ บางนาทีอาจไม่มี trade แต่มีการปรับ quote หลายครั้ง

การคำนวณผลตอบแทนจาก trade ทุกคู่จึงผสมการเคลื่อนของมูลค่ากับผลของ bid–ask spread ตัวอย่างมูลค่าแฝงอยู่ที่ 100 คงที่ แต่ราคา trade สลับ 99.99 กับ 100.01 จะให้ผลตอบแทนสลับบวกและลบ ทั้งที่ราคาแฝงไม่ได้เปลี่ยน การเห็น short-lag correlation ติดลบในข้อมูลแบบนี้ยังไม่ใช่กลยุทธ์กำไรหลังหัก spread

ก่อนคำนวณต้องเลือกว่าใช้ trade, midpoint หรือ quote ฝั่งใด จัดการ time zone และ daylight saving ให้ตรง เก็บเครื่องหมายบอกข้อมูล stale และแยกช่วงตลาดเปิดกับปิด การใช้ราคาล่าสุดเติมในนาทีที่ไม่มี trade อาจเพิ่มผลตอบแทนศูนย์จำนวนมาก ส่วน tick size ทำให้ราคาขยับเป็นขั้นแทนค่าต่อเนื่อง

Stylized facts ที่เห็นในรายวันยังพบในข้อมูลระหว่างวันได้ แต่ microstructure มีอิทธิพลมากขึ้นตามความถี่ การดู dependence ยังต้องคำนึงถึงเวลาในวัน เพราะนาทีใกล้เปิดตลาดกับนาทีกลางวันอาจมีสเกลความผันผวนต่างกันซ้ำทุกวัน

## จังหวะในวันและข่าวที่ออกตามเวลา

หลายตลาดมี volatility สูงใกล้เปิดหรือปิดตลาด และมีจุดสูงขึ้นรอบข่าวเศรษฐกิจบางรายการ รูปแบบขึ้นกับตลาด ตารางซื้อขาย และช่วงที่ศึกษา ตลาด FX ตลอดวันยังมีช่วงที่ผู้ค้าจากหลายภูมิภาคทำงานทับกัน จึงไม่ควรใช้รูป U หรือเวลาออกข่าวชุดเดียวกับทุกตลาด

ตัวอย่างในงานของ Taylor แยกผลของเวลาเปิดตลาด ข่าวเศรษฐกิจ และช่วงที่ตลาดต่างประเทศเปิดซ้อนกัน เวลาที่บันทึกในตัวอย่างเก่าเป็นข้อมูลของตลาดและช่วงศึกษานั้น การเปรียบเทียบข้ามประเทศต้องจัดการช่วงที่เปลี่ยน daylight saving คนละวันด้วย

ให้ \(v_j\) เป็น variance ของผลตอบแทนช่วง j ที่ประมาณจากหลายวัน สัดส่วน variance ภายในช่วงเปิดตลาดคือ

$$
a_j=\frac{v_j}{\sum_{k=1}^{M}v_k},\qquad \sum_{j=1}^{M}a_j=1.
$$

ถ้าผลตอบแทนแต่ละช่วงไม่มี covariance ต่อกัน และ variance รวมของช่วงเปิดตลาดวันเป้าหมายเท่ากับ h เมื่อใช้ seasonal profile นี้ จะจัด variance ให้ช่วง j เท่ากับ \(h a_j\) ส่วน SD ของช่วงนั้นเท่ากับ \(\sqrt h\sqrt{a_j}\) ตัวคูณ variance กับตัวคูณ SD จึงต่างกัน หากผลตอบแทนระหว่างช่วงมี covariance ต้องนับพจน์เหล่านั้นด้วยก่อนตีความผลรวมเป็น variance ของผลตอบแทนทั้งช่วงเปิดตลาด



สร้าง profile สมมติจากเส้นลดลงหลังเปิดตลาด เส้นเพิ่มขึ้นก่อนปิด และส่วนที่สูงขึ้นใกล้ช่วงที่ 31 แล้ว normalize ให้แต่ละชุดรวมเป็น 100% เส้นนี้แสดงการจัดสัดส่วนเท่านั้น ไม่ได้อ้างว่า variance รวมของวันข่าวเท่ากับวันอื่น และเวลาในภาพไม่ใช่ตารางประกาศข่าวจริง

ความเคลื่อนไหวหลังข่าวขึ้นกับส่วนที่ต่างจากความคาดหวัง ไม่ใช่เพียงตัวเลขที่ประกาศหรือจำนวนพาดหัวข่าว หากจะศึกษาผลของข่าว ต้องเก็บ announcement time, ค่าที่ประกาศ และค่าคาดการณ์ที่มีอยู่ก่อนข่าวให้ตรงกัน ดูตัวอย่างการแยก news surprise ใน [Andersen, Bollerslev, Diebold และ Vega (2003)](https://public.econ.duke.edu/~boller/research.html)

In [7]:
for news in [False,True]:
    profile = intraday_profile(news)
    close(sum(profile),1)
    daily_sd = .01
    interval_sd = [daily_sd*math.sqrt(a) for a in profile]
    close(sum(s*s for s in interval_sd),daily_sd**2)
    print(f"News bump={news}: largest share {max(profile):.6%}; sum {sum(profile):.6%}")
print("Variance weights sum to one; SD multipliers are their square roots.")

News bump=False: largest share 3.553526%; sum 100.000000%
News bump=True: largest share 4.719755%; sum 100.000000%
Variance weights sum to one; SD multipliers are their square roots.


## Realized variance รวมการแกว่งที่ราคาปิดซ่อนไว้

ให้ \(r_{t,j}\) เป็น log return ระหว่างสองราคาที่เก็บต่อกันในวัน t ใช้ N ช่วงย่อย เรานิยาม

$$
\operatorname{RV}_t=\sum_{j=1}^{N}r_{t,j}^2,\qquad
\operatorname{RVol}_t=\sqrt{\operatorname{RV}_t}.
$$

[Realized variance](../glossary.html#realized-variance) มีหน่วยเป็นผลตอบแทนยกกำลังสอง ส่วน realized volatility หรือ realized SD มีหน่วยเดียวกับผลตอบแทน ชื่อ RV ในงานบางชิ้นใช้เรียกไม่เหมือนกัน จึงควรอ่านนิยามก่อนเทียบตัวเลข บทนี้สงวน RV ไว้สำหรับผลรวมกำลังสอง

ตัวอย่าง log returns ภายในวันเป็น 1%, −1%, 1%, −1% จะรวมได้ศูนย์ ราคาจึงกลับมาที่เดิม แต่ \(\operatorname{RV}=4(0.01)^2=0.0004\) และ \(\sqrt{\operatorname{RV}}=2\%\) การยกกำลังสองผลตอบแทนต้น–ปลายวันจะให้ศูนย์ เพราะไม่เห็นการแกว่งระหว่างทาง



สองเส้นมีผลตอบแทนต้น–ปลายช่วงเท่ากับศูนย์ โดยมีสี่ผลตอบแทนย่อยเท่ากัน การคำนวณ √RV ใช้ค่าทศนิยมก่อนแปลงเป็นเปอร์เซ็นต์

สำหรับ log price ที่เป็น continuous semimartingale และไม่มี measurement noise เมื่อเก็บถี่ขึ้น RV จะลู่เข้า quadratic variation ซึ่งเท่ากับ integrated variance \(\int\sigma_s^2ds\) ของช่วงนั้น หากมี jumps ขีดจำกัดจะรวมกำลังสองของขนาด jump ด้วย

$$
\operatorname{RV}_t\ \longrightarrow\
\int_t^{t+1}\sigma_s^2\,ds+\sum_{t<s\leq t+1}(\Delta\log P_s)^2.
$$

ความเชื่อมโยงกับ quadratic variation อธิบายไว้ในบท [Applied Stochastic Calculus](../applied-stochastic-calculus.html) ส่วนการใช้ realized measures ประมาณและพยากรณ์ volatility พัฒนาต่อใน [Andersen, Bollerslev, Diebold และ Labys (2003)](https://econ.duke.edu/~boller/Published_Papers/ecta_03.pdf)

หากข้อมูลครอบคลุมเฉพาะตลาดเปิด RV ก็ครอบคลุมเฉพาะช่วงนั้น การคูณ \(\sqrt{252}\) เช่น 1% เป็นประมาณ 15.87% ต่อปี เป็นการแปลงหน่วยด้วยสมมติฐาน scaling และจำนวนวัน ไม่ได้เติม overnight risk ให้เอง การใส่ overnight return ยกกำลังสองเพิ่มหนึ่งค่าช่วยนับช่วงที่หายไปบางส่วน แต่ไม่ได้ให้ความละเอียดเท่าการมีราคาตลอดคืน

In [8]:
for scale in [.01,.002]:
    log_prices = [0]
    for r in [scale,-scale,scale,-scale]:
        log_prices.append(log_prices[-1]+r)
    stats = realized_variance(log_prices)
    close(log_prices[-1],0)
    close(stats['volatility'],2*scale)
    print(f"Four returns +/-{scale:.2%}: RV={stats['variance']:.7f}, sqrt(RV)={stats['volatility']:.4%}, closing return=0")
print(f"1% session SD * sqrt(252) = {.01*math.sqrt(252):.4%}; excludes overnight")

Four returns +/-1.00%: RV=0.0004000, sqrt(RV)=2.0000%, closing return=0
Four returns +/-0.20%: RV=0.0000160, sqrt(RV)=0.4000%, closing return=0
1% session SD * sqrt(252) = 15.8745%; excludes overnight


## ราคาถี่ขึ้นมีทั้งข้อมูลเพิ่มและ noise เพิ่ม

ให้ \(p_j^*\) เป็น log price แฝง และราคาที่สังเกตเป็น \(p_j=p_j^*+\epsilon_j\) โดย \(\epsilon_j\) แทน [microstructure noise](../glossary.html#microstructure-noise) สมมติ noise มีค่าเฉลี่ยศูนย์ variance \(\eta^2\) เป็นอิสระข้ามเวลาและจากราคาแฝง จะได้ observed return

$$
r_j=\Delta p_j^*+\epsilon_j-\epsilon_{j-1}.
$$

ผลต่าง noise มี variance \(2\eta^2\) ดังนั้นบน sampling grid ที่มี N ช่วง

$$
\mathbb E[\operatorname{RV}_{\rm observed}]
=\mathbb E[\operatorname{RV}_{\rm latent}]+2N\eta^2.
$$

เมื่อเก็บราคาถี่ขึ้น N เพิ่ม ส่วน bias จาก noise จึงเพิ่มตามในแบบจำลองนี้ ถ้า latent returns ไม่มี serial covariance ส่วน noise ยังทำให้ covariance ของผลตอบแทนติดกันเท่ากับ \(-\eta^2\) ได้ งานเรื่องการเลือก sampling frequency จึงต้องพิจารณาทั้งความละเอียดและ microstructure noise ดู [Aït-Sahalia, Mykland และ Zhang, *How Often to Sample a Continuous-Time Process…*](https://www.nber.org/papers/w9611)

เมื่อ noise เป็นศูนย์ การเก็บถี่ขึ้นช่วยประมาณ integrated variance ภายใต้สมมติฐานของโมเดลได้ แต่ RV บนเส้นทางเดียวไม่จำเป็นต้องขยับเข้าหาค่าจริงทีละขั้นอย่างสม่ำเสมอ เมื่อ noise ไม่เป็นศูนย์ การเลือกข้อมูลถี่ที่สุดก็อาจเพิ่มความผิดพลาด ยังมีวิธีอย่าง subsampling, realized kernels และ pre-averaging สำหรับจัดการปัญหานี้ โดยแต่ละวิธีมีเงื่อนไขของตนเอง

In [9]:
for noise_bps in [0,3,10]:
    data = intraday_sample(noise_bps)
    for stride in [1,5,15,30,390]:
        latent = realized_variance(data['latent'],stride)
        observed = realized_variance(data['observed'],stride)
        bias = 2*observed['count']*data['eta']**2
        if noise_bps==0:
            close(latent['variance'],observed['variance'])
        print(f"Noise={noise_bps} bps, every {stride} min: N={observed['count']}, latent SD={latent['volatility']:.5%}, observed SD={observed['volatility']:.5%}, expected noise RV={bias:.8f}")
close(2*390*.0003**2,.0000702)
close(2*78*.0003**2,.00001404)
print("Observed-minus-latent RV of one realization need not equal its expectation.")

Noise=0 bps, every 1 min: N=390, latent SD=1.00759%, observed SD=1.00759%, expected noise RV=0.00000000
Noise=0 bps, every 5 min: N=78, latent SD=0.99700%, observed SD=0.99700%, expected noise RV=0.00000000
Noise=0 bps, every 15 min: N=26, latent SD=1.08954%, observed SD=1.08954%, expected noise RV=0.00000000
Noise=0 bps, every 30 min: N=13, latent SD=1.11371%, observed SD=1.11371%, expected noise RV=0.00000000
Noise=0 bps, every 390 min: N=1, latent SD=1.51420%, observed SD=1.51420%, expected noise RV=0.00000000
Noise=3 bps, every 1 min: N=390, latent SD=1.00759%, observed SD=1.31379%, expected noise RV=0.00007020
Noise=3 bps, every 5 min: N=78, latent SD=0.99700%, observed SD=1.06703%, expected noise RV=0.00001404
Noise=3 bps, every 15 min: N=26, latent SD=1.08954%, observed SD=1.10030%, expected noise RV=0.00000468
Noise=3 bps, every 30 min: N=13, latent SD=1.11371%, observed SD=1.08931%, expected noise RV=0.00000234
Noise=3 bps, every 390 min: N=1, latent SD=1.51420%, observed SD=1

## หารด้วย volatility แล้วเหลืออะไร

เมื่อรวมช่วง volatility สูงกับต่ำ raw returns อาจมีหางหนามาก การหารด้วย volatility ของแต่ละช่วงช่วยลดความต่างของสเกล งาน empirical บางชุดพบว่า realized volatility มีการแจกแจงเบ้ขวา และ standardized returns เข้าใกล้ Normal มากขึ้น แต่ผลนี้ขึ้นกับสินทรัพย์ estimator และช่วงตัวอย่าง

ถ้าใช้ \(\sqrt{\operatorname{RV}_t}\) ซึ่งคำนวณจากราคาตลอดวัน t ตัวหารจะรู้ได้เมื่อจบวัน การดู \(r_t/\sqrt{\operatorname{RV}_t}\) จึงเป็นการวิเคราะห์ย้อนหลัง หากต้องการตั้ง VaR ก่อนวันเริ่ม ต้องใช้ \(\widehat\sigma_{t\mid t-1}\) ที่ประมาณได้จากข้อมูลก่อนหน้านั้น

$$
z_t^{\rm forecast}=\frac{r_t-\widehat\mu_{t\mid t-1}}{\widehat\sigma_{t\mid t-1}}.
$$

จากนั้นตรวจ distribution และ ACF ของทั้ง \(z_t\) กับ \(z_t^2\) เพื่อดูว่าโมเดลอธิบายความสัมพันธ์ที่ต้องการได้แค่ไหน นอกจากนี้ผลตอบแทนและ RV ต้องครอบคลุมช่วงเวลาเดียวกัน การหาร close-to-close return ด้วย open-to-close realized volatility จะมี overnight component อยู่ในตัวเศษเพียงด้านเดียว

ACF ของ volatility ที่ลดลงช้าเป็นลักษณะ persistence ที่ควรตรวจต่อ แต่กราฟ ACF เส้นเดียวแยก true long memory ออกจาก structural breaks หรือการผสมหลายสภาวะได้ไม่เด็ดขาด และการเห็น log volatility ดูใกล้ Normal ก็ยังไม่พิสูจน์ว่า volatility เป็น Lognormal ทุกช่วงเวลา

In [10]:
# Here sigma is KNOWN because we constructed the simulation. This is not a forecast.
known_sigma = [.005 if (i//50)%2==0 else .025 for i in range(600)]
standardized = [r/s for r,s in zip(returns,known_sigma)]
print("Known-sigma standardized simulation:",moments(standardized))
print(f"ACF of squared standardized shocks, lag 1={acf([z*z for z in standardized])[1]:.6f}")
print("In market data, same-day realized volatility is known only after observing that day.")

Known-sigma standardized simulation: {'mean': -0.040632624930982665, 'sd': 0.971290017192985, 'variance': 0.941831957002918, 'kurtosis': 2.8061421741970656}
ACF of squared standardized shocks, lag 1=-0.048319
In market data, same-day realized volatility is known only after observing that day.


## เหตุการณ์สั้น ๆ ที่ข้อมูลรายวันมองไม่เห็น

Flash Crash วันที่ 6 พฤษภาคม 2010 เป็นตัวอย่างที่ราคาสินทรัพย์สหรัฐฯ เคลื่อนลงและฟื้นกลับภายในวัน การดูราคาปิดวันเดียวจะสูญเสียรายละเอียดของเส้นทาง รายงานร่วม [CFTC และ SEC (2010)](https://www.sec.gov/news/studies/2010/marketevents-report.pdf) ใช้ข้อมูลธุรกรรมและการทำงานของตลาดเพื่อศึกษาลำดับเหตุการณ์ ไม่ได้อาศัย daily return เพียงค่าเดียว

ข้อมูลความถี่สูงยังช่วยตรวจการเปลี่ยนแปลงรวดเร็วที่อาจเป็น price jump แต่จุดโดดในกราฟอาจมาจากข้อมูลผิด quote ที่ล้าสมัย หรือ spread ที่กว้างขึ้นด้วย การสรุปว่ามี jump ต้องอาศัยการตรวจข้อมูลและวิธีทดสอบที่คำนึงถึง noise กับ sampling frequency

ในแบบจำลองที่มี jumps RV จะนับกำลังสองของ jumps รวมอยู่ด้วย หากโจทย์ต้องการแยก continuous variation ออกจาก jump variation ต้องใช้ estimator เพิ่ม เช่น bipower variation ภายใต้เงื่อนไขของวิธีนั้น แทนการเรียก RV ทั้งก้อนว่า diffusion variance

สำหรับผู้จัดการกองทุน ข้อมูลระหว่างวันจึงช่วยทั้งวัดความเสี่ยงที่เกิดขึ้นจริงและตรวจว่าช่วงใดสร้างความเสียหาย ส่วนการพยากรณ์วันถัดไปต้องประเมินกับข้อมูลนอกช่วงฝึก โดยเทียบระยะเวลาพยากรณ์และ information set เดียวกัน

## ทดลองและตรวจคำตอบ

1. ซื้อหุ้นที่ 100 ปลายช่วงราคา 99 และได้รับปันผล 2 คำนวณ simple return กับ log return แล้วอธิบายว่าทำไมใช้ price return อย่างเดียวจึงได้เครื่องหมายต่างกัน
2. ในตัวทดลอง clustering สับลำดับวันแล้วเทียบ mean, SD, kurtosis และ ACF ของ |r| สถิติใดเปลี่ยน และสถิติใดเก็บข้อมูลเกี่ยวกับลำดับเวลาไว้?
3. เลื่อนอัตราส่วน SD ของ mixture เป็น 1 แล้วคำนวณ kurtosis จากสูตร ลองอธิบายว่าถ้าสุ่มเลือกสภาวะใหม่ทุกวันอย่างอิสระ หางหนายังอยู่ได้โดยไม่มี clustering อย่างไร
4. คำนวณ RV และ √RV จาก log returns 1%, −1%, 1%, −1% แล้วเทียบกับการเก็บเฉพาะราคาต้น–ปลายวัน
5. เมื่อ noise ของ log price เป็น ±3 bps อย่างอิสระ จงหาส่วนเพิ่มของ E[RV] ถ้าใช้ 390 ผลตอบแทน เทียบกับ 78 ผลตอบแทน โดยไม่เปลี่ยนระยะเวลาของวัน

**เปิดแนวคำตอบ**

ข้อ 1 simple return เท่ากับ 1% และ log return เท่ากับ \(\log(1.01)\approx0.9950\%\) ส่วนราคาดิบลด 1% เพราะยังไม่ได้รวมปันผล

ข้อ 2 mean, SD, kurtosis และ histogram เท่าเดิมทุกค่า การสับลำดับเปลี่ยนคู่ข้อมูลที่ใช้หา ACF จึงเปลี่ยนสถิติที่วัดความสัมพันธ์ข้ามเวลา

ข้อ 3 เมื่อ SD เท่ากัน variance ของ \(\sigma^2\) เป็นศูนย์ จึงได้ κ=3 ส่วนการสุ่มสภาวะแบบอิสระยังสร้าง mixture marginal ได้ แต่ไม่ได้สร้าง persistence ของสภาวะ

ข้อ 4 RV=0.0004 และ √RV=2% ส่วนผลตอบแทนต้น–ปลายวันเป็นศูนย์

ข้อ 5 \(\eta=0.0003\) ให้ \(2(390)\eta^2=0.0000702\) กับ \(2(78)\eta^2=0.00001404\) ในหน่วยทศนิยม² ต่างกันห้าเท่า ตัวเลขนี้เป็นส่วนเพิ่มของค่าคาดหมายภายใต้โมเดล noise ไม่ใช่ความต่างที่ต้องเกิดพอดีทุกเส้นทาง

[ดาวน์โหลด Python Notebook](asset-returns-stylized-facts.ipynb) เพื่อคำนวณ returns, ACF, Box–Pierce/Ljung–Box, mixture kurtosis, intraday profile และ realized variance ใช้ seed และวิธีคำนวณเดียวกับตัวทดลอง พร้อมภาพที่ฝังไว้ในไฟล์ ใช้ Python standard library ได้โดยไม่ต้องดาวน์โหลดราคาตลาด

## อ่านเพิ่มเติม

- Stephen J. Taylor, *Asset Price Dynamics, Volatility, and Prediction* (2005), บท 2, 4 และ 12 · [บทนำจาก Princeton University Press](https://assets.press.princeton.edu/chapters/i8055.pdf)
- Benoit Mandelbrot, [*The Variation of Certain Speculative Prices* (1963)](https://oftp.cyrax.hu/doc/mandelbrot.pdf), โดยเฉพาะข้อสังเกตเรื่องการเกิดกลุ่มของความผันผวนในหน้า 418
- NIST, [*Autocorrelation Plot*](https://www.itl.nist.gov/div898/handbook/eda/section3/eda331.htm), นิยาม sample ACF และกรอบอ้างอิง
- Andersen, Bollerslev, Diebold และ Labys, [*Modeling and Forecasting Realized Volatility* (2003)](https://econ.duke.edu/~boller/Published_Papers/ecta_03.pdf)
- Aït-Sahalia, Mykland และ Zhang, [*How Often to Sample a Continuous-Time Process in the Presence of Market Microstructure Noise*](https://www.nber.org/papers/w9611), working paper 2003, ตีพิมพ์ในปี 2005
- CFTC และ SEC, [*Findings Regarding the Market Events of May 6, 2010*](https://www.sec.gov/news/studies/2010/marketevents-report.pdf), รายงานวันที่ 30 กันยายน 2010